# 10 Machine logs + photos: a random forest on each, then blend them

**Questions this notebook answers**
1. How well does a **random forest on the machine logs alone** predict the defect, and which log features matter (ablation)?
2. Can we **blend** the logs model with the photo model from notebook 09, and which way of blending works best?

**Why the two should help each other.** The logs describe the *cause side* (settings, nozzle wear, alarms, time since maintenance), known before the part is inspected. The photo shows the *effect* (what the edge looks like). A worn nozzle in the logs and dross blobs in the photo are two views of the same event. In the multimodal knowledge graph both hang off the same Job node. This notebook measures whether combining them is better than either alone.

**Three ways to blend two models** (standard names in multimodal learning; Baltrušaitis et al. 2019 survey, "Multimodal machine learning: a survey and taxonomy", IEEE TPAMI):

| Name | How | Like |
|---|---|---|
| **Early fusion** | put log and photo features side by side in one table, train one model | one expert reading both files |
| **Late fusion** | train one model per source, average their predicted probabilities | two experts voting |
| **Stacking** | train one model per source, then a small "judge" model learns how much to trust each, per defect | a manager who has learned which expert is right about what |

**Inputs:** `data/processed/master_job_table.csv` (notebook 03) and `data/processed/photo_features.csv` (notebook 09). Run 01, 03 and 09 first.
**Outputs:** `data/processed/logs_ablation.csv`, `data/processed/fusion_results.csv`, figures in `data/processed/figures/`.

## 0. Setup

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import f1_score, accuracy_score, classification_report
warnings.filterwarnings("ignore", category=UserWarning)   # sklearn warns when a rare class is missing from a fold

OUT = Path("..") / "data" / "processed"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
CLASSES = ["ok", "dross", "incomplete_cut", "rough_cut", "bevel_deviation"]

master = pd.read_csv(OUT / "master_job_table.csv")
photo = pd.read_csv(OUT / "photo_features.csv")
print(len(master), "jobs in the master table;", len(photo), "photos measured")

## 1. The log features, in groups
Notebook 03 built these columns. For the ablation we sort them into four groups by *what they describe*:

| Group | Columns | Physical meaning |
|---|---|---|
| settings | thickness, power, feed, gas pressure, focus, nozzle size, stand-off, bevel angle, parts, material, gas | what the program asked for |
| program | `feed_vs_table`, `is_fast`, `is_bevel`, `late_shift` | how far the program departs from the cutting table, and when |
| wear & maintenance | `NozzleOperatingHours`, `hours_since_glass_clean`, `hours_since_collision` | the state of the machine |
| alarms | `alarms_<code>` counts during the job | what the machine complained about |

In [ ]:
LOG_GROUPS = {
    "settings": ["Thickness_mm", "LaserPower_W", "FeedRate_mm_min", "GasPressureSet_bar", "FocusPosition_mm",
                 "NozzleDiameter_mm", "Standoff_mm", "BevelAngle_deg", "PartsOnSheet"]
                + [c for c in master.columns if c.startswith(("mat_", "gas_"))],
    "program": ["feed_vs_table", "is_fast", "is_bevel", "late_shift"],
    "wear & maintenance": ["NozzleOperatingHours", "hours_since_glass_clean", "hours_since_collision"],
    "alarms": [c for c in master.columns if c.startswith("alarms_")],
}
LOG = [c for g in LOG_GROUPS.values() for c in g]
assert set(LOG) == set(master.columns) - {"JobID", "StartTime", "defect"}, "a master-table column is in no group"

# photo features from notebook 09, WITHOUT tilt_off: tilt_off already uses the job's bevel angle (a log value),
# so leaving it out keeps the two sources separate and the blending test fair
PHOTO = ["sharpness", "brightness", "light_gradient", "face_height_px", "tilt_deg",
         "dark_frac_lower", "dark_frac_upper", "dark_band_height", "blob_count", "blob_mean_size",
         "below_edge_dark", "bottom_edge_ragged", "top_edge_ragged",
         "stripes_per100px_top", "stripe_contrast_top", "stripe_straight_top",
         "stripes_per100px_low", "stripe_contrast_low", "stripe_straight_low"]
print(len(LOG), "log features,", len(PHOTO), "photo features")

## 2. Random forest on the machine logs alone
Same model and same scoring as notebook 09, so the numbers can be compared: random forest, `class_weight="balanced"`, **5-fold cross-validation repeated 3 times**, **macro F1** (every defect counts equally).

Notebook 03 used gradient boosting on the same table. A random forest is a different way to combine trees (many independent trees that vote, instead of trees that each fix the last one's mistakes); comparing them is a small extra check.

In [ ]:
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
forest = lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1)

def score(model, X, y):
    r = cross_validate(model, X, y, cv=CV, scoring=["f1_macro", "accuracy"])
    return {"macro_F1": r["test_f1_macro"].mean(), "F1_sd": r["test_f1_macro"].std(), "accuracy": r["test_accuracy"].mean(), "jobs": len(y)}

logs_all = master[master["defect"].isin(CLASSES)].reset_index(drop=True)
y_all = logs_all["defect"]
print(f"{len(logs_all)} labelled jobs; always-'ok' accuracy {(y_all == 'ok').mean():.1%}")
full_logs = score(forest(), logs_all[LOG], y_all)
print(f"logs random forest: macro F1 {full_logs['macro_F1']:.3f} ± {full_logs['F1_sd']:.3f}, accuracy {full_logs['accuracy']:.1%}")

pred = cross_val_predict(forest(), logs_all[LOG], y_all, cv=StratifiedKFold(5, shuffle=True, random_state=42))
print(classification_report(y_all, pred, labels=CLASSES, digits=2))

**How to read it:** look at the per-defect F1. The logs should be good where a log value *is* the cause (a bevel job, a `_FAST` program) and weak where the cause is hidden or the same settings sometimes give a good part and sometimes not. That's normal: the logs say what *might* go wrong, not what did.

## 3. Ablation on the log features
Same idea as notebook 09: drop one group, or keep one group only, and re-score.

In [ ]:
log_results = {"FULL: all log features": full_logs}
for g, cols in LOG_GROUPS.items():
    log_results[f"without {g}"] = score(forest(), logs_all[[c for c in LOG if c not in cols]], y_all)
for g, cols in LOG_GROUPS.items():
    log_results[f"only {g}"] = score(forest(), logs_all[cols], y_all)

log_abl = pd.DataFrame(log_results).T
log_abl["change_vs_full"] = log_abl["macro_F1"] - full_logs["macro_F1"]
log_abl = log_abl[["macro_F1", "F1_sd", "change_vs_full", "accuracy", "jobs"]]
log_abl.to_csv(OUT / "logs_ablation.csv")
display(log_abl.style.format({"macro_F1": "{:.3f}", "F1_sd": "{:.3f}", "change_vs_full": "{:+.3f}", "accuracy": "{:.1%}", "jobs": "{:.0f}"}))

order = log_abl.sort_values("macro_F1")
plt.figure(figsize=(8, 0.4 * len(order) + 1))
plt.barh(order.index, order["macro_F1"], xerr=order["F1_sd"], color=["tab:red" if i.startswith("FULL") else "tab:blue" for i in order.index])
plt.axvline(full_logs["macro_F1"], color="tab:red", ls="--", lw=1); plt.xlim(0, 1.02)
plt.xlabel("macro F1 (5-fold × 3 cross-validation)"); plt.title("Machine-log model ablation")
plt.tight_layout(); plt.savefig(FIG / "logs_ablation.png", dpi=150); plt.show()

**How to read it:** a group whose removal drops the score is doing work; a group that scores well *alone* carries most of the signal. Compare with notebook 03's EDA: the causes you found (nozzle wear, dirty glass, gas supply, `_FAST` programs, head collision) should show up as the groups that matter.

## 4. Blend logs and photos
For a fair comparison every model below is scored on **the same jobs** (labelled jobs that also have a sharp photo) and **the same folds**. The blended models pick their columns with a `ColumnTransformer`, so each base model only ever sees its own source, and everything is fitted inside the training folds (no peeking at test jobs).

Stacking's judge is a **logistic regression** trained on the base models' *out-of-fold* probabilities (the base models' guesses on jobs they didn't train on), so the judge learns how much to trust each model on new jobs, not on jobs it memorised.

In [ ]:
both = master.merge(photo[["job_id", "blurry"] + PHOTO], left_on="JobID", right_on="job_id")
both = both[both["defect"].isin(CLASSES) & ~both["blurry"]].reset_index(drop=True)
y = both["defect"]
print(len(both), "jobs with a label and a sharp photo")

take = lambda cols: ColumnTransformer([("cols", "passthrough", cols)])
logs_model = make_pipeline(take(LOG), forest())
photo_model = make_pipeline(take(PHOTO), forest())
X = both[LOG + PHOTO]

fusion = {
    "logs only": score(logs_model, X, y),
    "photo only": score(photo_model, X, y),
    "early fusion (one forest on both)": score(forest(), X, y),
    "late fusion (average the two forests)": score(VotingClassifier([("logs", logs_model), ("photo", photo_model)], voting="soft"), X, y),
    "stacking (judge learns whom to trust)": score(StackingClassifier([("logs", logs_model), ("photo", photo_model)],
                                                    final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced"), cv=5), X, y),
}
# for reference: notebook 09's hand-made link between photo and log (tilt compared with the programmed bevel angle)
ref = both.merge(photo[["job_id", "tilt_off"]], on="job_id")
fusion["photo + tilt_off (knowledge-based link, notebook 09)"] = score(forest(), ref[PHOTO + ["tilt_off"]], ref["defect"])

fus = pd.DataFrame(fusion).T[["macro_F1", "F1_sd", "accuracy", "jobs"]]
fus.to_csv(OUT / "fusion_results.csv")
display(fus.style.format({"macro_F1": "{:.3f}", "F1_sd": "{:.3f}", "accuracy": "{:.1%}", "jobs": "{:.0f}"}))

plt.figure(figsize=(8, 3.6))
o = fus.sort_values("macro_F1")
plt.barh(o.index, o["macro_F1"], xerr=o["F1_sd"], color="tab:blue"); plt.xlim(0, 1.02)
plt.xlabel("macro F1 (same jobs, same folds)"); plt.title("Logs, photos, and three ways to blend them")
plt.tight_layout(); plt.savefig(FIG / "fusion_comparison.png", dpi=150); plt.show()

### Which defects does each source get right?
Per-defect F1 shows *where* the blend helps. One shuffled 5-fold split, the same for every model.

In [ ]:
cv1 = StratifiedKFold(5, shuffle=True, random_state=42)
per_class = {}
for name, model in [("logs", logs_model), ("photo", photo_model), ("early fusion", forest()),
                    ("stacking", StackingClassifier([("logs", logs_model), ("photo", photo_model)],
                                                    final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced"), cv=5))]:
    p = cross_val_predict(model, X, y, cv=cv1)
    per_class[name] = f1_score(y, p, labels=CLASSES, average=None)
per_class = pd.DataFrame(per_class, index=CLASSES).round(2)
per_class

### How much weight should the logs get? (late-fusion weight sweep)
Late fusion averages the two models' probabilities with weight *w* on the logs and 1 − *w* on the photo. *w* = 0 is photo only, *w* = 1 is logs only. **Caution:** the best *w* here is picked on the same jobs we score on, so its score is a little optimistic. Stacking is the honest way to learn the weights.

In [ ]:
p_logs = cross_val_predict(logs_model, X, y, cv=cv1, method="predict_proba")
p_photo = cross_val_predict(photo_model, X, y, cv=cv1, method="predict_proba")
labels = np.array(sorted(CLASSES))
ws = np.linspace(0, 1, 21)
f1s = [f1_score(y, labels[(w * p_logs + (1 - w) * p_photo).argmax(axis=1)], average="macro") for w in ws]
plt.figure(figsize=(6, 3.2)); plt.plot(ws, f1s, marker="o")
plt.xlabel("weight on the logs model (w)"); plt.ylabel("macro F1"); plt.title("Late fusion: how much to trust the logs")
plt.tight_layout(); plt.savefig(FIG / "fusion_weight_sweep.png", dpi=150); plt.show()
print(f"best w = {ws[int(np.argmax(f1s))]:.2f} (macro F1 {max(f1s):.3f}); w = 0.5 gives {f1s[10]:.3f}")

## 5. A harder test: train on the past, predict the future
Cross-validation mixes old and new jobs. In a factory you train on what happened and predict what comes next. Like notebook 03, we train on the first two thirds of the jobs by start time and test on the last third. Defects that only appear late (bevel_deviation in October) can't be learned this way, so macro F1 is taken over the defects present in the test set.

In [ ]:
both_t = both.assign(StartTime=pd.to_datetime(both["StartTime"], format="ISO8601")).sort_values("StartTime").reset_index(drop=True)
cut = int(len(both_t) * 2 / 3)
tr, te = both_t.iloc[:cut], both_t.iloc[cut:]
print("train", tr["StartTime"].min().date(), "to", tr["StartTime"].max().date(), "|", len(tr), "jobs")
print("test ", te["StartTime"].min().date(), "to", te["StartTime"].max().date(), "|", len(te), "jobs")
print("defects in training:", sorted(tr["defect"].unique()), "| in test:", sorted(te["defect"].unique()))

present = sorted(te["defect"].unique())
time_rows = {}
for name, model in [("logs only", logs_model), ("photo only", photo_model), ("early fusion", forest()),
                    ("stacking", StackingClassifier([("logs", logs_model), ("photo", photo_model)],
                                                    final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced"), cv=5))]:
    p = model.fit(tr[LOG + PHOTO], tr["defect"]).predict(te[LOG + PHOTO])
    time_rows[name] = {"macro_F1": f1_score(te["defect"], p, labels=present, average="macro"), "accuracy": accuracy_score(te["defect"], p)}
time_tab = pd.DataFrame(time_rows).T
time_tab.to_csv(OUT / "fusion_time_split.csv")
print(f"always-'ok' accuracy on the test period: {(te['defect'] == 'ok').mean():.1%}")
time_tab.style.format({"macro_F1": "{:.3f}", "accuracy": "{:.1%}"})

## 6. What to write in the report
- **Logs alone** score well below photos. That's expected: the logs record *conditions*, and the same conditions sometimes give a good part. Their value is that they explain the *cause*, which a photo can't.
- **Blending:** check whether early fusion and stacking beat the photo alone, and on which defects (section 4 per-defect table). Late fusion with equal weights can be *worse* than the photo alone, because it gives the weaker model half the vote; stacking fixes that by learning the weights.
- **The best "blend" may be a knowledge-based one.** Compare the `photo + tilt_off` row with the generic blends: one feature that encodes a physical rule (compare the tilt with the *programmed* bevel angle) can do more than feeding all 27 log columns to a model. That's the argument for a knowledge graph: it stores *which* log value relates to *which* photo evidence.
- **Past → future:** report this alongside the cross-validation score; it's the more honest number.
- **Caveats:** synthetic data; photo features designed after looking at the photos; only jobs with both a label and a sharp photo are in the blend comparison (in a real factory many jobs have no photo, and then the logs model is all you have).